# B1.5 · Vulnerability auditing: three generations of SAST

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.4 · Strategic planning and agent allocation](https://spbreed.github.io/cyber-commons/lessons/B1.4.html)**.

| | |
|---|---|
| Open-source tooling | OpenGrep, Semgrep OSS, CodeQL |
| Open-weight models | GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Stage 7 — Vulnerability auditing.** The deep-dive analysis stage, and the one
people think of as "SAST". It has had three generations, and knowing what each
can and cannot see is what stops you buying the wrong one.

**Generation 1 — grep.** Pattern-match dangerous constructs. Fast, zero setup,
fires on every occurrence whether reachable or not. Precision is poor, so it gets
muted.

**Generation 2 — rules with dataflow.** Semgrep, CodeQL, OpenGrep. Parse to an
AST or graph and track *taint*: does untrusted input reach a dangerous sink?
Precision improves enormously. The cost is that a rule only finds the pattern
someone wrote it for.

**Generation 3 — model review.** An open-weight model reads the code and reasons.
No rule needs to exist first, which is exactly its value — and it also invents
bugs that are not there, confidently.

The mistake is treating generation 3 as a replacement for generation 2. The
combination that works: rules for what rules do well, deterministically; the
model for what rules cannot express; and everything the model says treated as a
**hypothesis** until stages 8–12 confirm it.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

> **About the model in this notebook.** It runs offline against a deterministic
> stand-in so the lesson executes on a Kaggle kernel with no network. The
> stand-in is not a language model and is labelled as such wherever it appears.
> To run the identical pipeline stage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Generation 1 — grep, and why it gets muted

The safe functions in this corpus matter more than the buggy ones: a scanner that fires on parameterised SQL is one nobody runs twice.

In [ ]:
CODE = {
"db.py": '''
def get_user(conn, name):
    # BUG: user input concatenated into SQL
    return conn.execute("SELECT * FROM users WHERE name = \'" + name + "\'")

def get_user_safe(conn, name):
    # parameterised — the driver escapes it
    return conn.execute("SELECT * FROM users WHERE name = ?", (name,))

def audit_note(conn, msg):
    # a constant string. No user input anywhere.
    return conn.execute("INSERT INTO audit(msg) VALUES (\'startup\')")
''',
"ops.py": '''
import os, subprocess

def ping(host):
    # BUG: shell string built from user input
    os.system("ping -c1 " + host)

def ping_safe(host):
    subprocess.run(["ping", "-c1", host], check=True)
''',
"files.py": '''
def read_doc(base, filename):
    # BUG: path joined from untrusted input
    return open(base + "/" + filename).read()
''',
}
import re
GREP_RULES = [("CWE-89","SQL injection",r"execute\("),
              ("CWE-78","command injection",r"os\.system|subprocess"),
              ("CWE-22","path traversal",r"open\(")]
def gen1(code):
    return [(cwe, name, f, i, ln.strip())
            for f, src in code.items()
            for i, ln in enumerate(src.splitlines(), 1)
            for cwe, name, pat in GREP_RULES if re.search(pat, ln)]

g1 = gen1(CODE)
print(f"generation 1 (grep): {len(g1)} findings")
for cwe, name, f, i, ln in g1:
    print(f"   {cwe:8s}{f}:{i:<3} {ln[:52]}")

In [ ]:
TRUTH = {("CWE-89","db.py",4), ("CWE-78","ops.py",6), ("CWE-22","files.py",4)}
def score(findings, label):
    got = {(c, f, i) for c, _, f, i, _ in findings}
    tp, fp, fn = len(got & TRUTH), len(got - TRUTH), len(TRUTH - got)
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    print(f"{label:32s} tp={tp} fp={fp} fn={fn}  precision={prec:.2f} recall={rec:.2f}")
    return prec, rec
score(g1, "generation 1 · grep")
print("\nfalse positives:")
for cwe, name, f, i, ln in g1:
    if (cwe, f, i) not in TRUTH: print(f"   {f}:{i:<3} {ln[:56]}")

## 3 · Generation 2 — taint rules

The improvement is not a better pattern. It is a different question: *does untrusted input reach this sink?* A function parameter is untrusted; a string literal is not.

In [ ]:
import ast
class TaintRule:
    SINKS = {"execute": ("CWE-89","SQL injection"),
             "system":  ("CWE-78","command injection"),
             "open":    ("CWE-22","path traversal")}
    def scan(self, fname, src):
        out = []
        for fn in [n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef)]:
            tainted = {a.arg for a in fn.args.args}
            for call in [n for n in ast.walk(fn) if isinstance(n, ast.Call)]:
                sink = (call.func.attr if isinstance(call.func, ast.Attribute)
                        else getattr(call.func, "id", ""))
                if sink not in self.SINKS: continue
                cwe, name = self.SINKS[sink]
                for arg in call.args:
                    if self._concat_taint(arg, tainted):
                        out.append((cwe, name, fname, call.lineno,
                                    ast.get_source_segment(src, call) or ""))
                        break
        return out
    @staticmethod
    def _concat_taint(node, tainted):
        for n in ast.walk(node):
            if isinstance(n, ast.BinOp) and isinstance(n.op, ast.Add):
                if {x.id for x in ast.walk(n) if isinstance(x, ast.Name)} & tainted:
                    return True
        return False

rule = TaintRule()
g2 = [f for n, s in CODE.items() for f in rule.scan(n, s)]
print(f"generation 2 (taint rules): {len(g2)} findings")
for cwe, name, f, i, snip in g2: print(f"   {cwe:8s}{f}:{i:<3} {snip[:52]}")
print()
score(g2, "generation 2 · taint rules")

## 4 · Generation 3 — what rules structurally cannot see

Generation 2 is perfect on this corpus. So why involve a model? Because a rule only finds what someone wrote it for. Here is a bug with no rule: an authorization check that is *present* and wrong.

In [ ]:
CODE["authz.py"] = '''
def can_delete(user, doc):
    # Reads "or" where it means "and". No sink, no taint, no pattern.
    if user.is_admin or user.id == doc.owner_id or doc.is_public:
        return True
    return False
'''
print("generation 1 on authz.py:", gen1({"authz.py": CODE["authz.py"]}) or "nothing")
print("generation 2 on authz.py:", rule.scan("authz.py", CODE["authz.py"]) or "nothing")

class StandIn:
    """DETERMINISTIC STAND-IN — not a language model. See the note above."""
    KNOWN = {
     "authz.py": [{"cwe":"CWE-863","line":4,"confidence":0.82,
                   "rationale":"disjunctive permission check: a non-public document "
                               "owned by another user is deletable whenever is_public "
                               "is true, and delete rights are never checked"}],
     "db.py": [{"cwe":"CWE-89","line":4,"confidence":0.95,
                "rationale":"name is concatenated into the query string"},
               {"cwe":"CWE-89","line":12,"confidence":0.41,
                "rationale":"audit_note also calls execute"}],     # HALLUCINATION
    }
    def review(self, fname, src): return self.KNOWN.get(fname, [])

model = StandIn()
print("\ngeneration 3 (model review):")
for fname in ("authz.py", "db.py"):
    for f in model.review(fname, CODE[fname]):
        print(f"   {f['cwe']:9s}{fname}:{f['line']:<3} conf={f['confidence']:.2f}  "
              f"{f['rationale'][:52]}")
print("\nIt found the authorization bug neither earlier generation can see.")
print("It also invented a SQL injection in a function with a constant string.")

In [ ]:
# Stage 7 output: rules + gated model hypotheses. Confirmation is stages 8-12.
GATE = 0.70
def stage7(code, rule, model, gate=GATE):
    findings, suppressed = [], []
    for fname, src in code.items():
        for cwe, name, f, i, snip in rule.scan(fname, src):
            findings.append({"src":"rules","cwe":cwe,"file":f,"line":i,
                             "confidence":1.0,"status":"confirmed-by-rule"})
        for m in model.review(fname, src):
            row = {"src":"model","cwe":m["cwe"],"file":fname,"line":m["line"],
                   "confidence":m["confidence"],"status":"HYPOTHESIS"}
            (findings if m["confidence"] >= gate else suppressed).append(row)
    seen, dedup = set(), []
    for f in sorted(findings, key=lambda r: r["src"]):
        k = (f["cwe"], f["file"], f["line"])
        if k in seen: continue
        seen.add(k); dedup.append(f)
    return dedup, suppressed

final, suppressed = stage7(CODE, rule, model)
print(f"stage 7 emits {len(final)} findings, {len(suppressed)} suppressed below {GATE}")
for f in final:
    print(f"   [{f['src']:5s}] {f['cwe']:9s}{f['file']}:{f['line']:<3} "
          f"conf={f['confidence']:.2f}  {f['status']}")
TRUTH_FULL = TRUTH | {("CWE-863","authz.py",4)}
got = {(f["cwe"], f["file"], f["line"]) for f in final}
print(f"\ntp={len(got & TRUTH_FULL)} fp={len(got - TRUTH_FULL)} fn={len(TRUTH_FULL - got)}")
assert not (got - TRUTH_FULL) and not (TRUTH_FULL - got)
print("Every model finding is marked HYPOTHESIS. Stages 8-12 decide.")

## What you just proved

Grep produces 6 findings at 50% precision, flagging the parameterised query, the constant insert and the safe subprocess call. Taint rules find exactly the 3 real injection bugs at 100% precision and recall and find nothing in `authz.py`. The model finds the authorization bug at 0.82 confidence and hallucinates one SQL injection at 0.41. Stage 7 emits 4 findings with zero false positives, every model finding marked as a hypothesis.

## Your turn

Point the stand-in at a real GLM-4.6 or Kimi K2 through Ollama and run it on `authz.py` ten times. The variance in what it reports — and in its confidence — decides whether you can gate on confidence at all.

---

**Next → [B1.6 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*